In this file we will do some hypothesis testing and analyze datasets deeper by cross datasets

In [1]:
import numpy as np
import pandas as pd
import os

folder_path = os.getcwd()
dataset_folder_path_raw = os.path.join(folder_path,"..\\data_raw\\")
dataset_folder_path_clean = os.path.join(folder_path,"..\\data_cleaned\\")

In [3]:
orders_dataset = os.path.join(dataset_folder_path_clean,'orders_dataset.csv')
orders_df = pd.read_csv(orders_dataset)
orders_df.head(1)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18


In [7]:
orders_reviews_dataset = os.path.join(dataset_folder_path_raw,'order_reviews_dataset.csv')
order_reviews_df = pd.read_csv(orders_reviews_dataset)
order_reviews_df.head(1)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59


In [9]:
orders_reviews = orders_df.merge(
    order_reviews_df[['order_id', 'review_score']],
    on='order_id',
    how='inner'
)

orders_reviews['order_purchase_timestamp'] = pd.to_datetime(orders_reviews['order_purchase_timestamp'],errors='coerce')

orders_reviews['order_delivered_customer_date'] = pd.to_datetime(orders_reviews['order_delivered_customer_date'],errors='coerce')

orders_reviews['delivery_days'] = (orders_reviews['order_delivered_customer_date'] - orders_reviews['order_purchase_timestamp']).dt.days

In [10]:
# Average Delivery Time by Review Score
delivery_vs_review = (
    orders_reviews.groupby('review_score').agg(
                      avg_delivery_days=('delivery_days', 'mean'),
                      median_delivery_days=('delivery_days', 'median'),
                      order_count=('order_id', 'count')
                  ).reset_index()
                  .sort_values('review_score'))

print(delivery_vs_review)

   review_score  avg_delivery_days  median_delivery_days  order_count
0             1          20.704180                  16.0        11424
1             2          16.227460                  13.0         3151
2             3          13.704390                  11.0         8179
3             4          11.857403                  10.0        19142
4             5          10.233590                   9.0        57328


In [14]:
# Delivery Duration Distribution Bins vs Review Score
bins = [0, 5, 10, 15, 20, 30, 60, float('inf')]
labels = [
    '0-5', '5-10', '10-15',
    '15-20', '20-30',
    '30-60', '60+'
]

orders_reviews['delivery_days'] = pd.cut(
    orders_reviews['delivery_days'],bins=bins, labels=labels
)

distribution_table = (
    pd.crosstab(
        orders_reviews['review_score'],
        orders_reviews['delivery_days']
    )
)

print(distribution_table)

delivery_days   0-5   5-10  10-15  15-20  20-30  30-60  60+
review_score                                               
1               776   1529   1139    742   1235   1618  114
2               301    573    404    280    378    239   10
3               896   1783   1386    811    792    305   16
4              2509   4858   3396   1871   1354    257   25
5              9951  16171   9574   4339   2555    439   32


In [15]:
# Convert the crosstab to row percentages:
distribution_pct = (
    pd.crosstab(
        orders_reviews['review_score'],
        orders_reviews['delivery_days'],
        normalize='index'
    ) * 100
).round(2)

print(distribution_pct)

delivery_days    0-5   5-10  10-15  15-20  20-30  30-60   60+
review_score                                                 
1              10.85  21.38  15.92  10.37  17.27  22.62  1.59
2              13.78  26.22  18.49  12.81  17.30  10.94  0.46
3              14.96  29.77  23.14  13.54  13.22   5.09  0.27
4              17.58  34.04  23.80  13.11   9.49   1.80  0.18
5              23.11  37.55  22.23  10.08   5.93   1.02  0.07


In [16]:
# Status vs review scores
status_vs_review = pd.crosstab(
    orders_reviews['order_status'],
    orders_reviews['review_score']
)

print(status_vs_review)

review_score     1     2     3      4      5
order_status                                
approved         1     0     0      1      0
canceled       422    44    48     26     69
created          2     0     0      0      1
delivered     9406  2941  7961  18987  57066
invoiced       230    26    16     15     26
processing     256    18     9      6      7
shipped        644    79   110     87    123
unavailable    463    43    35     20     36


In [17]:
# In terms of %
status_vs_review_pct = pd.crosstab(
    orders_reviews['order_status'],
    orders_reviews['review_score'],
    normalize='index'
) * 100

status_vs_review_pct = status_vs_review_pct.round(2)

print(status_vs_review_pct)

review_score      1     2      3      4      5
order_status                                  
approved      50.00  0.00   0.00  50.00   0.00
canceled      69.29  7.22   7.88   4.27  11.33
created       66.67  0.00   0.00   0.00  33.33
delivered      9.76  3.05   8.26  19.70  59.22
invoiced      73.48  8.31   5.11   4.79   8.31
processing    86.49  6.08   3.04   2.03   2.36
shipped       61.74  7.57  10.55   8.34  11.79
unavailable   77.55  7.20   5.86   3.35   6.03


In [19]:
order_payments_dataset = os.path.join(dataset_folder_path_raw,'order_payments_dataset.csv')
order_payments_df = pd.read_csv(order_payments_dataset)
order_payments_df.head(1)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33


In [20]:
# Group by payment type
payment_summary = (
    order_payments_df.groupby('payment_type')
                     .agg(
                         total_value=('payment_value', 'sum'),
                         avg_value=('payment_value', 'mean'),
                         median_value=('payment_value', 'median'),
                         num_payments=('payment_value', 'count')
                     )
                     .sort_values('total_value', ascending=False)
                     .reset_index()
)

print(payment_summary)

  payment_type  total_value   avg_value  median_value  num_payments
0  credit_card  12542084.19  163.319021        106.87         76795
1       boleto   2869361.27  145.034435         93.89         19784
2      voucher    379436.87   65.703354         39.28          5775
3   debit_card    217989.79  142.570170         89.30          1529
4  not_defined         0.00    0.000000          0.00             3


In [21]:
total_payment = payment_summary['total_value'].sum()

payment_summary['percent_of_total'] = (
    payment_summary['total_value'] / total_payment * 100
).round(2)

print(payment_summary)

  payment_type  total_value   avg_value  median_value  num_payments  \
0  credit_card  12542084.19  163.319021        106.87         76795   
1       boleto   2869361.27  145.034435         93.89         19784   
2      voucher    379436.87   65.703354         39.28          5775   
3   debit_card    217989.79  142.570170         89.30          1529   
4  not_defined         0.00    0.000000          0.00             3   

   percent_of_total  
0             78.34  
1             17.92  
2              2.37  
3              1.36  
4              0.00  


In [22]:
# High value orders  per payment type
high_value_threshold = 1000

high_value_orders = order_payments_df[
    order_payments_df['payment_value'] > high_value_threshold
]

high_value_summary = high_value_orders.groupby('payment_type')['order_id'].count().reset_index()
high_value_summary = high_value_summary.rename(columns={'order_id':'high_value_count'})
print(high_value_summary)

  payment_type  high_value_count
0       boleto               178
1  credit_card               944
2   debit_card                15
3      voucher                13


In [24]:
order_items_dataset = os.path.join(dataset_folder_path_clean,'order_items_dataset.csv')
order_items_df = pd.read_csv(order_items_dataset)
order_items_df.head(1)

,Unnamed: 0,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29


In [25]:
products_dataset = os.path.join(dataset_folder_path_clean,'products_dataset.csv')
products_df = pd.read_csv(products_dataset)
products_df.head(1)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery,40.0,287.0,1.0,225.0,16.0,10.0,14.0


In [26]:
# Product category vs sales volume
order_items_products = order_items_df.merge(
    products_df[['product_id', 'product_category_name']],
    on='product_id',
    how='left'
)

category_sales_volume = (
    order_items_products.groupby('product_category_name')['order_item_id']
    .count()  # count of items sold
    .reset_index(name='items_sold')
    .sort_values('items_sold', ascending=False)
)

print(category_sales_volume)

        product_category_name  items_sold
7              bed_bath_table       11115
43              health_beauty        9670
67             sports_leisure        8641
39            furniture_decor        8334
15      computers_accessories        7827
..                        ...         ...
11          cds_dvds_musicals          14
52                 la_cuisine          14
59                   pc_gamer           9
29  fashion_childrens_clothes           8
63      security_and_services           2

[73 rows x 2 columns]


In [27]:
# Category vs sales price
category_sales_price = (
    order_items_products.groupby('product_category_name').agg(
        avg_price=('price', 'mean'),
        median_price=('price', 'median'),
        min_price=('price', 'min'),
        max_price=('price', 'max'),
        num_items=('price', 'count')
    )
    .sort_values('avg_price', ascending=False)
    .reset_index()
)

print(category_sales_price)

                    product_category_name    avg_price  median_price  \
0                               computers  1098.340542      1100.000   
1   small_appliances_home_oven_and_coffee   624.285658       587.000   
2                       home_appliances_2   476.124958       225.945   
3              agro_industry_and_commerce   342.124858       258.650   
4                     musical_instruments   281.616000        94.835   
..                                    ...          ...           ...   
68                             food_drink    54.602446        38.745   
69                      cds_dvds_musicals    52.142857        45.000   
70                    diapers_and_hygiene    40.194615        37.000   
71                                flowers    33.637576        25.990   
72                         home_comfort_2    25.342333        12.900   

    min_price  max_price  num_items  
0       34.50    6729.00        203  
1       10.19    2899.00         76  
2       13.90    2350

In [40]:
customers_dataset = os.path.join(dataset_folder_path_raw,'customers_dataset.csv')
customers_df = pd.read_csv(customers_dataset)
customers_df.head(1)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP


In [41]:
sellers_dataset = os.path.join(dataset_folder_path_raw,'sellers_dataset.csv')
sellers_df = pd.read_csv(sellers_dataset)
sellers_df.head(1)

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP


### Demand and Supply analysis

In [48]:
# Convert dates
orders_df['order_purchase_timestamp'] = pd.to_datetime(orders_df['order_purchase_timestamp'])

# -----------------------------
# 2. Merge datasets
# -----------------------------
df = orders_df.merge(order_items_df, on='order_id', how='inner') \
           .merge(customers_df[['customer_id', 'customer_state']], on='customer_id', how='left') \
           .merge(sellers_df[['seller_id', 'seller_state']], on='seller_id', how='left') \
           .merge(products_df[['product_id', 'product_category_name']], on='product_id', how='left')

# Create year-month for trend analysis
df['year_month'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)

# -----------------------------
# 3. DEMAND by state
# -----------------------------
demand_state = df.groupby(['customer_state', 'year_month']).agg(
    total_orders=('order_id', 'nunique'),
    total_items_demand=('order_item_id', 'count'),
    unique_products_demand=('product_id', 'nunique')
).reset_index()

# -----------------------------
# 4. SUPPLY by state (seller side)
# -----------------------------
supply_state = df.groupby(['seller_state', 'year_month']).agg(
    active_sellers=('seller_id', 'nunique'),
    total_items_supply=('order_item_id', 'count')
).reset_index()

# -----------------------------
# 5. Combine supply and demand by state
# -----------------------------

state_analysis = demand_state.merge(
    supply_state,
    left_on='customer_state',
    right_on='seller_state',
    how='left',
    suffixes=('_demand', '_supply')
)


# Drop redundant seller_state column
state_analysis.drop(columns=['seller_state'], inplace=True)

# -----------------------------
# 6. Metrics
# -----------------------------
state_analysis['supply_gap'] = (state_analysis['total_items_supply'] - state_analysis['total_items_demand'])

state_analysis['demand_supply_ratio'] = (state_analysis['total_items_demand'] / state_analysis['total_items_supply'])

# Replace inf / NaN
state_analysis.replace([float('inf'), -float('inf')], pd.NA, inplace=True)


# -----------------------------
# 7. Sort insights
# -----------------------------
high_demand_low_supply = state_analysis.sort_values('supply_gap').head(10)

high_supply_states = state_analysis.sort_values('total_items_supply',ascending=False).head(10)

high_demand_states = state_analysis.sort_values('total_items_demand',ascending=False).head(10)

# -----------------------------
# 8. Output
# -----------------------------
print("🔴 States with highest unmet demand (supply deficit):",high_demand_low_supply.value_counts())

print("\n🟢 Top demand states:",high_demand_states.value_counts())
print("\n🔵 Top supply states:\n",high_supply_states.value_counts())
# print(high_supply_states)

🔴 States with highest unmet demand (supply deficit): customer_state  year_month_demand  total_orders  total_items_demand  unique_products_demand  year_month_supply  active_sellers  total_items_supply  supply_gap  demand_supply_ratio
SP              2018-05            3197          3685                2267                    2018-09            1.0             1.0                 -3684.0     3685.000000            1
                2018-08            3218          3652                2469                    2018-09            1.0             1.0                 -3651.0     3652.000000            1
                2018-04            3056          3551                2265                    2018-09            1.0             1.0                 -3550.0     3551.000000            1
                2018-05            3197          3685                2267                    2016-10            87.0            209.0               -3476.0     17.631579              1
                2018-03    

In [53]:
# Create year-month for trend analysis
df['year_month'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)

# -----------------------------
# 3. DEMAND by state
# -----------------------------
demand_state = df.groupby(['customer_state', 'year_month']).agg(
    total_orders=('order_id', 'nunique'),
    total_items_sold=('order_item_id', 'count'),
    unique_products_demand=('product_id', 'nunique')
).reset_index()

# -----------------------------
# 4. SUPPLY by state (seller side)
# -----------------------------
supply_state = df.groupby(['seller_state', 'year_month']).agg(
    active_sellers=('seller_id', 'nunique'),
    total_fulfilled_items=('order_item_id', 'count')
).reset_index()

# -----------------------------
# 5. Combine supply and demand by state
# -----------------------------
# Match states: simple left merge on year_month; some states may differ
state_analysis = demand_state.merge(
    supply_state,
    left_on=['customer_state', 'year_month'],
    right_on=['seller_state', 'year_month'],
    how='left',
    suffixes=('_demand', '_supply')
)

# Drop redundant seller_state column
state_analysis.drop(columns=['seller_state'], inplace=True)

# -----------------------------
# 6. Supply-Demand gap metrics
# -----------------------------
state_analysis['demand_supply_ratio'] = (
    state_analysis['total_items_sold'] / state_analysis['total_fulfilled_items']
)
state_analysis['supply_gap'] = (
    state_analysis['total_fulfilled_items'] - state_analysis['total_items_sold']
)

state_analysis['demand_growth'] = state_analysis.groupby('customer_state')['total_items_sold'].pct_change()
state_analysis['supply_growth'] = state_analysis.groupby('customer_state')['total_fulfilled_items'].pct_change()

# -----------------------------
# 7. Top mismatch states
# -----------------------------
top_gap_states = state_analysis.sort_values('supply_gap').head(10)

# -----------------------------
# 8. Output
# -----------------------------
print("Top 10 states with supply < demand (negative supply_gap):")
print(top_gap_states[['customer_state', 'year_month', 'total_items_sold', 'total_fulfilled_items', 'supply_gap']])

Top 10 states with supply < demand (negative supply_gap):
    customer_state year_month  total_items_sold  total_fulfilled_items  \
383             RJ    2017-11              1212                  424.0   
386             RJ    2018-02              1053                  307.0   
387             RJ    2018-03              1040                  345.0   
385             RJ    2018-01              1022                  345.0   
388             RJ    2018-04               964                  342.0   
382             RJ    2017-10               768                  194.0   
389             RJ    2018-05               971                  401.0   
384             RJ    2017-12               870                  319.0   
381             RJ    2017-09               700                  166.0   
379             RJ    2017-07               637                  108.0   

     supply_gap  
383      -788.0  
386      -746.0  
387      -695.0  
385      -677.0  
388      -622.0  
382      -574.0  
3